<a href="https://colab.research.google.com/github/Coobaltoo13/milling-tool-life-fusion/blob/main/02_autoencoder_mlp_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Escenario 2: Autoencoder + MLP + LSTM

**Proyecto:** Fusión Multimodal de Vibración y Corriente para Predicción de Vida Útil en Fresado

**Objetivo:** Evaluar si un autoencoder mejora la predicción al limpiar las señales antes de pasarlas al modelo MLP + LSTM.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow:", tf.__version__)

TensorFlow: 2.20.0


In [11]:
url = 'https://raw.githubusercontent.com/Coobaltoo13/milling-tool-life-fusion/main/data/raw/FeatureAndMetadata_Milling.csv'
df_raw = pd.read_csv(url, sep=';', header=None, dtype=str) # Read everything as string initially

# The second row of df_raw (index 1) contains the actual column names
column_names = df_raw.iloc[1].tolist()
df = df_raw[2:].copy() # Use .copy() to avoid SettingWithCopyWarning
df.columns = column_names
df = df.reset_index(drop=True)

# Identify all columns that should be numeric (features + target)
# Assume the first 121 columns in the raw header are intended features, but filter out 'FileName' if present.
potential_feature_cols = df.columns[:121].tolist()
feature_cols = [col for col in potential_feature_cols if col not in ['FileName', 'CycleToFailure']]

target_col = 'CycleToFailureNormalized'

# Combine feature and target columns for numeric conversion
cols_to_process = feature_cols + [target_col]

for col in cols_to_process:
    if col in df.columns: # Check if column exists in the DataFrame
        # Add .strip() to remove leading/trailing whitespaces before replacement and conversion
        df[col] = df[col].str.strip().str.replace(',', '.', regex=False).apply(pd.to_numeric, errors='coerce')

# Now select X and y based on the corrected feature_cols and target_col
X_df = df[feature_cols]
y_series = df[target_col]

# Convert to numpy arrays
X = X_df.values
y = y_series.values

# Apply mask only to X and y, where NaNs should not exist
mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
X = X[mask]
y = y[mask]

# --- Debugging NaN values in the final X and y before masking (should be minimal/none for X_df, y_series) ---
print("NaNs in X_df before final mask:")
nan_in_X_df = X_df.isnull().sum()
print(nan_in_X_df[nan_in_X_df > 0])
print("NaNs in y_series before final mask:")
nan_in_y_series = y_series.isnull().sum()
print(nan_in_y_series[nan_in_y_series > 0])
print(f"Number of rows dropped by final mask: {len(df) - len(X)}")
# ----------------------------

print("X:", X.shape)
print("y:", y.shape)

NaNs in X_df before final mask:
Series([], dtype: int64)
NaNs in y_series before final mask:
[]
Number of rows dropped by final mask: 0
X: (968, 120)
y: (968,)


No hay datos basura. Los 968 ciclos son válidos.

X y y coinciden. Ambos tienen 968 elementos.

No se perdió nada. El dataset está completo.

In [13]:
#Dividir train/test ANTES de escalar
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (774, 120) (774,)
Test: (194, 120) (194,)


In [15]:
#Normalizar
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Datos normalizados")
print("Media train:", X_train_scaled.mean().round(4))
print("Desviación train:", X_train_scaled.std().round(4))


Datos normalizados
Media train: 0.0
Desviación train: 1.0


## 1. Autoencoder

El autoencoder comprime las 120 características en un vector de 16 números (representación latente) y luego intenta reconstruir la señal original. Si el error de reconstrucción es bajo, significa que aprendió una representación limpia y útil.

In [16]:
#Definir y entrenar autoencoder
input_dim = X_train_scaled.shape[1]
latent_dim = 16

# Encoder
encoder_input = layers.Input(shape=(input_dim,))
x = layers.Dense(64, activation='relu')(encoder_input)
x = layers.Dense(32, activation='relu')(x)
latent = layers.Dense(latent_dim, activation='relu', name='latent')(x)

# Decoder
x = layers.Dense(32, activation='relu')(latent)
x = layers.Dense(64, activation='relu')(x)
decoder_output = layers.Dense(input_dim, activation='linear')(x)

# Autoencoder completo
autoencoder = models.Model(encoder_input, decoder_output)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

# Entrenar
history_ae = autoencoder.fit(
    X_train_scaled, X_train_scaled,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         7,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent (Dense)                  │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 120)            │         7,800 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,808 (81.28 KB)

 Trainable params: 20,808 (81.28 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 1.0104 - val_loss: 0.6105
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.8616 - val_loss: 0.4508
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6776 - val_loss: 0.3513
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.5541 - val_loss: 0.3120
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.4742 - val_loss: 0.2823
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4125 - val_loss: 0.2550
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.3707 - val_loss: 0.2386
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.3420 - val_loss: 0.2251
Epoch 9/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3171 - val_loss: 0.2174
Epoch 10/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.3002 - val_loss: 0.2088
Epoch 11/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2874 - val_loss: 0.2002
Epoch 12/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.2682 - val_l

El autoencoder logró comprimir las 120 características a solo 16 números, y aun así puede reconstruir las señales con un error bajo (~0.12). Eso significa que la representación latente es útil y captura lo importante de las señales.

In [17]:
#Evaluar reconstruccion
X_test_reconstructed = autoencoder.predict(X_test_scaled)
mse_reconstruction = np.mean((X_test_scaled - X_test_reconstructed) ** 2)

print(f"Error de reconstrucción (MSE): {mse_reconstruction:.4f}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
Error de reconstrucción (MSE): 0.1635



-Error de reconstrucción (MSE)	0.1635

-val_loss final (durante entrenamiento)	0.1258

La diferencia es normal: en test el modelo se enfrenta a datos que nunca vio, así que el error sube un poco.  0.1635 sigue siendo bajo y confirma que el autoencoder aprendió una representación útil de las señales.

## 2. Representación Latente

Usamos solo la parte encoder del autoencoder para transformar las señales originales en vectores de 16 números. Esa es la versión "limpia" que alimentará al MLP + LSTM.

In [18]:
# Modelo encoder: desde la entrada hasta la capa latente
encoder = models.Model(encoder_input, latent)

# Transformar train y test
X_train_latent = encoder.predict(X_train_scaled)
X_test_latent = encoder.predict(X_test_scaled)

print("Latente train:", X_train_latent.shape)
print("Latente test:", X_test_latent.shape)

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
Latente train: (774, 16)
Latente test: (194, 16)


Las 120 características originales se comprimieron a solo 16 números por ciclo. Eso es lo que alimentará al MLP + LSTM.

In [20]:
#Reformar para LSTM
 # LSTM necesita (muestras, pasos_tiempo, características)
X_train_lstm = X_train_latent.reshape((X_train_latent.shape[0], 1, latent_dim))
X_test_lstm = X_test_latent.reshape((X_test_latent.shape[0], 1, latent_dim))

print("Train LSTM:", X_train_lstm.shape)
print("Test LSTM:", X_test_lstm.shape)

Train LSTM: (774, 1, 16)
Test LSTM: (194, 1, 16)


## 3. MLP + LSTM

Ahora entrenamos el mismo tipo de modelo que en el Escenario 1, pero usando las 16 características latentes en lugar de las 120 originales.

In [21]:
model = models.Sequential([
    layers.Input(shape=(1, latent_dim)),

    # MLP inicial
    layers.TimeDistributed(layers.Dense(32, activation='relu')),

    # LSTM
    layers.LSTM(32, return_sequences=False),

    # MLP final
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='linear')
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

history = model.fit(
    X_train_lstm, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 1, 32)          │           544 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,409 (36.75 KB)

 Trainable params: 9,409 (36.75 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 7s 37ms/step - loss: 0.2734 - mae: 0.4260 - val_loss: 0.0826 - val_mae: 0.2428
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0660 - mae: 0.2179 - val_loss: 0.0493 - val_mae: 0.1834
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0497 - mae: 0.1811 - val_loss: 0.0428 - val_mae: 0.1680
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0438 - mae: 0.1670 - val_loss: 0.0395 - val_mae: 0.1587
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0400 - mae: 0.1583 - val_loss: 0.0371 - val_mae: 0.1516
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0361 - mae: 0.1487 - val_loss: 0.0353 - val_mae: 0.1467
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0335 - mae: 0.1435 - val_loss: 0.0333 - val_mae: 0.1399
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0317 - mae: 0.1377 - val_loss: 0.0324 - val_mae: 0.1391
Epoch 9/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0303 - mae:

Escenario 2 tiene menos overfitting: el train loss no baja a casi cero como en el Escenario 1, lo que significa que no se está memorizando los datos.

Pero el val_loss es más alto: en validación el Escenario 1 parecía mejor.

La clave: hay que ver qué pasa en test (datos que nunca vio). Eso es lo que realmente importa.

In [22]:
y_pred = model.predict(X_test_lstm).flatten()

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n" + "=" * 40)
print("RESULTADOS - ESCENARIO 2 (Autoencoder + MLP + LSTM)")
print("=" * 40)
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step

RESULTADOS - ESCENARIO 2 (Autoencoder + MLP + LSTM)
RMSE: 0.1593
MAE:  0.1148
R²:   0.7095


Conclusión: El autoencoder comprimió las señales a 16 dimensiones, pero perdió información relevante para predecir la vida útil.

El autoencoder fue entrenado para reconstruir las 120 características originales, no para predecir la vida útil. Al comprimir a 16 dimensiones, conservó lo necesario para reconstruir las señales, pero descartó información que era útil para predecir.